# Experiment No. 10: Single-Layer Artificial Neural Network (ANN)

**Aim:** To implement a single-layer ANN using Python and NumPy for binary classification of Iris flowers.

This notebook uses the `iris.data` file from the UCI Iris dataset. It classifies Iris-setosa as `0` and Iris-versicolor as `1`. The model is implemented with NumPy, sigmoid activation, and gradient descent.

## Step 1: Import libraries

NumPy is used for the ANN calculations, and Pandas is used to read the dataset.

In [1]:
import numpy as np
import pandas as pd

## Step 2: Load the Iris dataset

Your screenshot shows the `iris` folder inside Downloads. This path assumes your Windows username is `prana`. If your username or folder location is different, edit the path below.

In [2]:
file_path = r"C:\Users\prana\Downloads\iris\iris.data"
data = pd.read_csv(file_path, header=None)

data.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
data = data.dropna()

print(data.head())
print("Original dataset shape:", data.shape)

   sepal_length  sepal_width  petal_length  petal_width      species
0           5.1          3.5           1.4          0.2  Iris-setosa
1           4.9          3.0           1.4          0.2  Iris-setosa
2           4.7          3.2           1.3          0.2  Iris-setosa
3           4.6          3.1           1.5          0.2  Iris-setosa
4           5.0          3.6           1.4          0.2  Iris-setosa
Original dataset shape: (150, 5)


## Step 3: Select two classes for binary classification

A single sigmoid output neuron is used for binary classification. Setosa is labelled `0` and Versicolor is labelled `1`.

In [3]:
data = data[data["species"].isin(["Iris-setosa", "Iris-versicolor"])].copy()
data["species"] = data["species"].map({"Iris-setosa": 0, "Iris-versicolor": 1})

print(data["species"].value_counts())
print("Selected dataset shape:", data.shape)

species
0    50
1    50
Name: count, dtype: int64
Selected dataset shape: (100, 5)


## Step 4: Separate input features and target labels

`X` contains the four measurements. `y` contains the class label for each flower.

In [4]:
X = data[["sepal_length", "sepal_width", "petal_length", "petal_width"]].to_numpy(dtype=float)
y = data["species"].to_numpy(dtype=int).reshape(-1, 1)

print("Input shape:", X.shape)
print("Output shape:", y.shape)

Input shape: (100, 4)
Output shape: (100, 1)


## Step 5: Split into training and testing data

We use 80% of the samples for training and 20% for testing. The test data is kept separate so accuracy is measured on samples the model did not train on.

In [5]:
np.random.seed(42)
indices = np.random.permutation(len(X))
split = int(0.8 * len(X))

train_idx = indices[:split]
test_idx = indices[split:]

X_train_raw = X[train_idx]
X_test_raw = X[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

print("Training samples:", len(X_train_raw))
print("Testing samples:", len(X_test_raw))

Training samples: 80
Testing samples: 20


## Step 6: Normalize the features

The mean and standard deviation are calculated using only the training data. The same values are used to normalize the test data and any new flower.

In [6]:
X_mean = X_train_raw.mean(axis=0)
X_std = X_train_raw.std(axis=0)

# Avoid division by zero if a feature has no variation
X_std[X_std == 0] = 1

X_train = (X_train_raw - X_mean) / X_std
X_test = (X_test_raw - X_mean) / X_std

print("Feature means:", X_mean)
print("Feature standard deviations:", X_std)

Feature means: [5.41875 3.11125 2.77875 0.755  ]
Feature standard deviations: [0.61969625 0.48269912 1.43192997 0.56211654]


## Step 7: Initialize weights, bias, and training settings

There are four input features, so the network has four weights and one bias.

In [7]:
np.random.seed(42)
weights = np.random.randn(4, 1) * 0.01
bias = 0.0
learning_rate = 0.1
epochs = 5000

print("Initial weights:\n", weights)
print("Initial bias:", bias)

Initial weights:
 [[ 0.00496714]
 [-0.00138264]
 [ 0.00647689]
 [ 0.0152303 ]]
Initial bias: 0.0


## Step 8: Define the sigmoid activation function

The sigmoid function returns a value between 0 and 1. A threshold of 0.5 is used to decide the predicted class.

In [8]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

## Step 9: Train the single-layer ANN

During training, the network calculates predictions, measures the error, and updates the weights and bias using gradient descent.

In [9]:
for i in range(epochs):
    z = np.dot(X_train, weights) + bias
    predictions = sigmoid(z)
    error = predictions - y_train

    dw = np.dot(X_train.T, error) / len(X_train)
    db = np.mean(error)

    weights = weights - learning_rate * dw
    bias = bias - learning_rate * db

print("Training completed!")
print("Final weights:\n", weights)
print("Final bias:", bias)

Training completed!
Final weights:
 [[ 1.49931276]
 [-2.12027059]
 [ 2.80736481]
 [ 2.84992238]]
Final bias: 0.023118294194280586


## Step 10: Evaluate the model on test data

This accuracy is calculated using the separate test set, not the training data.

In [10]:
test_probabilities = sigmoid(np.dot(X_test, weights) + bias)
y_pred = (test_probabilities >= 0.5).astype(int)
accuracy = np.mean(y_pred == y_test) * 100

print("Actual test labels:")
print(y_test.flatten())
print("Predicted test labels:")
print(y_pred.flatten())
print("Test Accuracy:", round(accuracy, 2), "%")

Actual test labels:
[1 1 0 0 0 1 0 0 0 1 1 1 1 1 0 1 1 0 1 1]
Predicted test labels:
[1 1 0 0 0 1 0 0 0 1 1 1 1 1 0 1 1 0 1 1]
Test Accuracy: 100.0 %


## Step 11: Predict the class of a new flower

The measurements below are in this order: sepal length, sepal width, petal length, petal width. Use the saved training mean and standard deviation to normalize the new sample.

In [11]:
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])
new_flower_scaled = (new_flower - X_mean) / X_std
new_probability = sigmoid(np.dot(new_flower_scaled, weights) + bias)[0, 0]

if new_probability >= 0.5:
    print("Predicted flower: Iris-versicolor (class 1)")
else:z
    print("Predicted flower: Iris-setosa (class 0)")

print("Probability of Iris-versicolor:", round(float(new_probability), 6))

Predicted flower: Iris-setosa (class 0)
Probability of Iris-versicolor: 0.000345


## Conclusion

Thus, a single-layer Artificial Neural Network (ANN) was implemented using Python and NumPy for binary classification of Iris flowers. The sigmoid activation function was used to predict class labels. The network learned by updating weights and bias through gradient descent. The model was evaluated using a separate test dataset, and a new flower sample was classified successfully.